# 3.3 가치함수와 벨만 기대방정식 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter03_3_bellman_value.ipynb)

책 본문: [3.3 가치함수와 벨만 기대방정식](https://smhanlab.com/book-ml/kor/ml2/chapter03/3.html)

이 노트북은 3.3절의 핵심 — **벨만 기대방정식** — 을 숫자로 확인합니다:

1. **두 개의 2상태 MDP** 만들기 (결정적 / 확률적 정책).
2. **닫힌 형태**(무한 등비급수)로 \\(V\\)를 직접 계산.
3. **반복적 정책평가**("값이 안 바뀔 때까지 재귀식 대입")로 다시 계산 —
   두 방법이 **같은 값**으로 수렴하는 것을 확인.
4. **벨만방정식이 실제로 등호로 성립하는지** 각 상태마다 검산.
5. \\(V^\pi = \sum_a \pi(a|s)\,Q^\pi(s,a)\\) — **V와 Q가 서로를 정의**하는
   "동전의 양면"임을 확인.
6. **수렴 곡선**으로 "반복 k번 ≈ 미래 k스텝까지의 리턴" 직관 확인.

numpy/matplotlib만 씁니다 — 외부 데이터 다운로드 불필요, CPU만으로 충분.

## 0. 환경 준비

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

import os
gamma = 0.9
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print(f"gamma = {gamma}   그림 저장 위치: {IMG}")

numpy 2.4.6 | matplotlib 3.11.1
gamma = 0.9   그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 두 개의 2상태 MDP 만들기

**구조는 똑같다** — S0 ↔ S1이 서로 오간다. **다르다**는 것은:
- **결정적 MDP**: 각 상태에서 할 수 있는 행동이 하나(확률 1). S0에서 항상 보상 1,
  S1에서 항상 보상 2.
- **확률적 MDP**: S0에서 정책이 \\(a_0\\)(보상 1)를 1/2, \\(a_1\\)(보상 3)을 1/2
  확률로 고른다. S1에서 항상 보상 2.

표시 형식: `P[s][a] = [(prob, next_state), ...]`, `R[s][a] = 즉시 보상`,
`pi[s][a] = 정책이 a를 고를 확률`.

In [2]:
# P[s][a] = [(prob, next_state), ...]  (행동 a의 전이 분포, 각 상태당 1개)
# ---------- 결정적 2상태 MDP ----------
# S0 --(r=1)--> S1 ;  S1 --(r=2)--> S0   (각각 행동 하나, 확률 1)
det = dict(
    P=[[ [(1.0, 1)] ],           # S0: action 0 -> S1 (prob 1)
        [ [(1.0, 0)] ]],         # S1: action 0 -> S0 (prob 1)
    R=[[1.0], [2.0]],
    pi=[[1.0], [1.0]],           # 각 상태에서 그 하나뿐인 행동
)

# ---------- 확률적 2상태 MDP ----------
# S0: pi(a0)=1/2(보상 1), pi(a1)=1/2(보상 3), 둘 다 -> S1
# S1: pi(a0)=1 (보상 2) -> S0
sto = dict(
    P=[[ [(1.0, 1)], [(1.0, 1)] ],   # S0: a0 -> S1 ; a1 -> S1
        [ [(1.0, 0)] ]],             # S1: a0 -> S0
    R=[[1.0, 3.0], [2.0]],
    pi=[[0.5, 0.5], [1.0]],          # S0에서 a0:1/2, a1:1/2 ; S1에서 a0:1
)
print("결정적 MDP: 상태 2개, 매 상태 행동 1개(확률 1)")
print("확률적 MDP: S0에서 a0/a1 각 1/2, S1에서 a0 확률 1")
print(f"두 예 모두 전이확률의 합 = 1 (마르코프 전이).  gamma={gamma}")

결정적 MDP: 상태 2개, 매 상태 행동 1개(확률 1)
확률적 MDP: S0에서 a0/a1 각 1/2, S1에서 a0 확률 1
두 예 모두 전이확률의 합 = 1 (마르코프 전이).  gamma=0.9


## 2. 방법 1: 무한 등비급수의 **닫힌 형태**

**결정적 예**에서 리턴은 \\(1, 2, 1, 2, \dots\\)이 무한히 반복되는
등비급수다. \\(V(S_0) = \frac{R_0 + \gamma R_1}{1-\gamma^2}\\),
\\(V(S_1) = \frac{R_1 + \gamma R_0}{1-\gamma^2}\\).
**확률적 예**는 연립방정식 \\(V_0 = 2+\gamma V_1,\; V_1 = 2+\gamma V_0\\)
(\\(S_0\\)의 **기대** 보상 = \\(\tfrac12(1)+\tfrac12(3)=2\\))를 푼다.

In [3]:
# 결정적: 무한 등비급수
V0_det = (1.0 + gamma*2.0) / (1 - gamma**2)
V1_det = (2.0 + gamma*1.0) / (1 - gamma**2)
print("=== 결정적 MDP (닫힌 형태) ===")
print(f"V(S0) = (1 + 0.9*2)/(1-0.9^2) = {V0_det:.5f}")
print(f"V(S1) = (2 + 0.9*1)/(1-0.9^2) = {V1_det:.5f}")
print(f"  -> 두 상태의 가치가 다르다 (매 스텝 보상이 고정: S0=1, S1=2)")
print()
# 확률적: 연립방정식 V0 = 2 + g V1 ; V1 = 2 + g V0
# V0 = 2 + g(2 + g V0) = 2 + 2g + g^2 V0  =>  V0(1-g^2) = 2(1+g)
V0_sto = 2*(1 + gamma) / (1 - gamma**2)
V1_sto = 2 + gamma*V0_sto
print("=== 확률적 MDP (닫힌 형태, 1/2-1/2) ===")
print(f"V(S0) = 2(1+g)/(1-g^2) = {V0_sto:.5f}")
print(f"V(S1) = 2 + g*V(S0)    = {V1_sto:.5f}")
print(f"  -> 두 상태의 가치가 같다 (S0의 기대보상 2 = S1의 고정보상 2)")

=== 결정적 MDP (닫힌 형태) ===
V(S0) = (1 + 0.9*2)/(1-0.9^2) = 14.73684
V(S1) = (2 + 0.9*1)/(1-0.9^2) = 15.26316
  -> 두 상태의 가치가 다르다 (매 스텝 보상이 고정: S0=1, S1=2)

=== 확률적 MDP (닫힌 형태, 1/2-1/2) ===
V(S0) = 2(1+g)/(1-g^2) = 20.00000
V(S1) = 2 + g*V(S0)    = 20.00000
  -> 두 상태의 가치가 같다 (S0의 기대보상 2 = S1의 고정보상 2)


## 3. 방법 2: **반복적 정책평가** (값이 안 바뀔 때까지 대입)

벨만방정식 \\(V(s) = R(s,\pi(s)) + \gamma\,\mathbb{E}_{s'}[V(s')]\\)의
좌변을 우변으로 계속 갱신한다. \\(\gamma<1\\)이면 **고정점**으로
보장되어 수렴한다 — 다음 장 **정책평가**의 핵심 루프다.
`max_change`(각 상태의 최대 변화량)가 \\(\theta\\) 미만이면 멈춘다.

In [4]:
def policy_evaluation(P, R, pi, gamma, max_iter=1000, theta=1e-9, V0=None):
    # 반복적 정책평가 (동기식: 매 반복이 이전 V를 참조). (V, max_change, iters) 반환.
    n = len(P)
    V = list(V0) if V0 is not None else [0.0]*n
    for it in range(1, max_iter+1):
        V_new = list(V)
        max_change = 0.0
        for s in range(n):
            # E_{a~pi, s'~P}[ r + g V(s') ]  — 이전 V 참조
            e = 0.0
            for a, pa in enumerate(pi[s]):
                if pa == 0.0:
                    continue
                e += pa * (R[s][a] + gamma * sum(p * V[s2] for p, s2 in P[s][a]))
            max_change = max(max_change, abs(e - V[s]))
            V_new[s] = e
        V = V_new
        if max_change < theta:
            break
    return V, max_change, it

# 결정적 예: 0에서 시작
V_det, mc, iters = policy_evaluation(det["P"], det["R"], det["pi"], gamma)
print("=== 결정적 MDP (반복 대입, V=0에서 시작) ===")
print(f"수렴: {iters}회 반복, 마지막 max_change={mc:.2e}")
print(f"V(S0)={V_det[0]:.5f}  V(S1)={V_det[1]:.5f}")
print(f"닫힌 형태({V0_det:.5f}, {V1_det:.5f})와 비교:")
print(f"  |차이| = ({abs(V_det[0]-V0_det):.2e}, {abs(V_det[1]-V1_det):.2e})  -> 거의 0")
assert abs(V_det[0]-V0_det) < 1e-4 and abs(V_det[1]-V1_det) < 1e-4
print("  [OK] 반복 대입 == 닫힌 형태")

=== 결정적 MDP (반복 대입, V=0에서 시작) ===
수렴: 205회 반복, 마지막 max_change=9.26e-10
V(S0)=14.73684  V(S1)=15.26316
닫힌 형태(14.73684, 15.26316)와 비교:
  |차이| = (6.36e-09, 6.14e-09)  -> 거의 0
  [OK] 반복 대입 == 닫힌 형태


## 4. 벨만방정식이 **등호로** 성립하는지 검산

계산된 \\(V\\)를 넣고, 각 상태마다 **우변**
\\(R + \gamma\,\mathbb{E}[V(s')]\\)을 다시 계산해 **좌변 \\(V(s)\\)**과
비교한다. 벨만방정식의 "해"라면 이 둘이 (부동소수점 오차 범위에서)
같아야 한다 — 좌변=우변인 **고정점**임의 직접 확인.

In [5]:
def bellman_residual(V, P, R, pi, gamma, s):
    e = sum(pa * (R[s][a] + gamma * sum(p * V[s2] for p, s2 in P[s][a]))
            for a, pa in enumerate(pi[s]) if pa > 0)
    return abs(V[s] - e)

print("state |  V(s) (좌변)  |  Bellman 우변  |  잔차 |  성립?")
print("-"*60)
for name, md_ in [("결정적", det), ("확률적", sto)]:
    V, _, _ = policy_evaluation(md_["P"], md_["R"], md_["pi"], gamma)
    for s in range(len(V)):
        rhs = sum(pa*(md_["R"][s][a] + gamma*sum(p*V[s2] for p,s2 in md_["P"][s][a]))
                  for a, pa in enumerate(md_["pi"][s]) if pa > 0)
        res = bellman_residual(V, md_["P"], md_["R"], md_["pi"], gamma, s)
        print(f"{name:5s} S{s}  | {V[s]:8.4f} | {rhs:8.4f} | {res:.2e} | "
              f"{'OK' if res < 1e-6 else 'FAIL'}")
        assert res < 1e-6
print()
print("[OK] 모든 상태에서 벨만방정식 좌변 = 우변 (잔차 < 1e-6)")

state |  V(s) (좌변)  |  Bellman 우변  |  잔차 |  성립?
------------------------------------------------------------
결정적   S0  |  14.7368 |  14.7368 | 8.33e-10 | OK
결정적   S1  |  15.2632 |  15.2632 | 4.17e-10 | OK
확률적   S0  |  20.0000 |  20.0000 | 8.33e-10 | OK
확률적   S1  |  20.0000 |  20.0000 | 8.33e-10 | OK

[OK] 모든 상태에서 벨만방정식 좌변 = 우변 (잔차 < 1e-6)


## 5. \\(V^\pi = \sum_a \pi(a|s)\,Q^\pi(s,a)\\) — V와 Q가 서로를 정의

**확률적 예**에서 행동가치 \\(Q^\pi(s,a) = R(s,a) + \gamma\,\mathbb{E}[V(s')]\\)를
각 (상태,행동)마다 계산하고, 정책 확률로 **가중평균**한 것이
상태가치 \\(V^\pi(s)\\)와 같은지 확인한다 — "동전의 양면"의 구체적 모습.

In [6]:
V_sto, _, _ = policy_evaluation(sto["P"], sto["R"], sto["pi"], gamma)
print("=== 확률적 MDP: Q와 V의 관계 ===")
for s in range(2):
    total = 0.0
    parts = []
    for a, pa in enumerate(sto["pi"][s]):
        if pa == 0.0:
            continue
        Q = sto["R"][s][a] + gamma * sum(p*V_sto[s2] for p, s2 in sto["P"][s][a])
        total += pa * Q
        parts.append(f"pi(a{a})={pa} Q={Q:.3f}")
    print(f"S{s}: " + "  ".join(parts))
    print(f"    sum pi*Q = {total:.5f}   vs   V(S{s}) = {V_sto[s]:.5f}   "
          f"일치={'OK' if abs(total-V_sto[s])<1e-9 else 'FAIL'}")
    assert abs(total - V_sto[s]) < 1e-9
print()
print("  Q(S0,a0)=19, Q(S0,a1)=21 -> 0.5*19+0.5*21 = 20 = V(S0)  (보상 1과 3의")
print("  평균 2가 S1의 고정보상 2와 같아 두 상태 가치 모두 20이 된 것)")

=== 확률적 MDP: Q와 V의 관계 ===
S0: pi(a0)=0.5 Q=19.000  pi(a1)=0.5 Q=21.000
    sum pi*Q = 20.00000   vs   V(S0) = 20.00000   일치=OK
S1: pi(a0)=1.0 Q=20.000
    sum pi*Q = 20.00000   vs   V(S1) = 20.00000   일치=OK

  Q(S0,a0)=19, Q(S0,a1)=21 -> 0.5*19+0.5*21 = 20 = V(S0)  (보상 1과 3의
  평균 2가 S1의 고정보상 2와 같아 두 상태 가치 모두 20이 된 것)


## 6. 수렴 곡선: "반복 k번 ≈ 미래 k스텝까지의 리턴"

\\(V=0\\)에서 시작해 반복할수록 **더 먼 미래**가 반영된다. \\(\gamma^k\\)가
빠르게 0으로 줄어들어 수렴은 **기하급수적**이다. 결정적 예에서
\\(V(S_0)\\)가 닫힌해(14.737)로 접근하는 모습을 그림으로 확인한다.
(반복을 10번 하면 10스텝까지, 50번이면 50스텝까지의 리턴이 반영된다.)

In [7]:
# 반복 200회의 V(S0), V(S1) 추적 (결정적 예) — 동기식(이전 V 참조)
n = 2
V = [0.0]*n
traj0, traj1 = [], []
for it in range(1, 201):
    V_new = list(V)
    for s in range(n):
        e = det["R"][s][0] + gamma * sum(p*V[s2] for p, s2 in det["P"][s][0])
        V_new[s] = e
    V = V_new
    traj0.append(V[0]); traj1.append(V[1])

iters_x = np.arange(1, 201)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(iters_x, traj0, color="#1d4ed8", lw=2, label="V(S0)  -> 14.737")
ax.plot(iters_x, traj1, color="#dc2626", lw=2, label="V(S1)  -> 15.263")
ax.axhline(V0_det, color="#1d4ed8", ls=":", lw=1.2)
ax.axhline(V1_det, color="#dc2626", ls=":", lw=1.2)
ax.set_xlabel("Iteration (k)")
ax.set_ylabel("V(s)")
ax.set_title("Iterative policy evaluation: starting from V=0, converging geometrically to the closed-form solution")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch03_3_value_iteration_convergence.svg")
plt.show()
print(f"반복 1:  V(S0)={traj0[0]:.4f}  V(S1)={traj1[0]:.4f}  (즉시 보상만)")
print(f"반복 10: V(S0)={traj0[9]:.4f}  V(S1)={traj1[9]:.4f}")
print(f"반복 50: V(S0)={traj0[49]:.4f}  V(S1)={traj1[49]:.4f}")
print(f"반복 200: V(S0)={traj0[-1]:.4f}  V(S1)={traj1[-1]:.4f}  (닫힌해 도달)")
print(f"그림 저장: {IMG}/ch03_3_value_iteration_convergence.svg")

반복 1:  V(S0)=1.0000  V(S1)=2.0000  (즉시 보상만)
반복 10: V(S0)=9.5984  V(S1)=9.9412
반복 50: V(S0)=14.6609  V(S1)=15.1845
반복 200: V(S0)=14.7368  V(S1)=15.2632  (닫힌해 도달)
그림 저장: /home/smhan/book-ml/kor/src/images/ch03_3_value_iteration_convergence.svg


## 7. 초기값 무관성: 0에서든 100에서든 같은 고정점으로

"고정점이 유일하고 반복이 수렴한다"는 **수축** 보장의 구체적 모습 —
초기값을 \\(0\\)과 \\(100\\)으로 바꿔도 **동일한 값**으로 수렴한다.
(초기값에 따라 다른 값으로 수렴했다면 \\(\\gamma\\ge1\\)이거나 할인율
곱셈이 빠진 것이다.)

In [8]:
V_from0, _, i0 = policy_evaluation(det["P"], det["R"], det["pi"], gamma, V0=[0.0, 0.0])
V_from100, _, i100 = policy_evaluation(det["P"], det["R"], det["pi"], gamma, V0=[100.0, 100.0])
print(f"V=0   에서 시작 -> V = ({V_from0[0]:.5f}, {V_from0[1]:.5f})   ({i0}회)")
print(f"V=100 에서 시작 -> V = ({V_from100[0]:.5f}, {V_from100[1]:.5f})   ({i100}회)")
assert abs(V_from0[0]-V_from100[0]) < 1e-6 and abs(V_from0[1]-V_from100[1]) < 1e-6
print("[OK] 초기값과 무관하게 같은 고정점(=닫힌해)으로 수렴")

V=0   에서 시작 -> V = (14.73684, 15.26316)   (205회)
V=100 에서 시작 -> V = (14.73684, 15.26316)   (219회)
[OK] 초기값과 무관하게 같은 고정점(=닫힌해)으로 수렴


## 8. 정리

| 확인한 것 | 결론 |
|---|---|
| 닫힌 형태 vs 반복 대입 | **같은 값**(결정적 14.737/15.263, 확률적 20/20) |
| 벨만방정식 | 계산된 V가 **고정점**(좌변=우변, 잔차 < 1e-6) |
| \\(V = \sum \pi Q\\) | 확률적 예에서 **동전의 양면** 성립 (19, 21 -> 20) |
| 수렴 | \\(\gamma<1\\)이면 **기하급수적**, 초기값 **무관** |

이 "한 스텝짜리 재귀식"이 다음 장 **동적계획법**의 정책평가·정책반복,
그리고 Chapter 4의 **최적**방정식(\\(\max\\)이 붙은 버전)으로 이어진다.
벨만방정식을 "어떻게 계산/학습하는가"만 바꾸면 **Q-학습, DQN, 정책
그래디언트**까지 만들어진다 — 이번 학기의 공통 뼈대다.